# ROSALIA MIMIC-ILS — IoU Regression Analysis

**Question:** Does *report-language uncertainty* lower ROSALIA's segmentation IoU **after controlling for lesion type and mask size**?

Group means alone are confounded: cardiomegaly is both the easiest lesion to segment *and* almost always described with certain language, so a raw certain-vs-uncertain gap can just be a disease-mix artifact. Least-squares regression estimates the effect of `is_uncertain` **holding disease and size constant**.

This notebook only needs the per-pair results CSV produced by the ROSALIA reproduction notebook (`rosalia_mimic_ils_per_pair.csv`). It does **not** re-run the model.

**What to look for:** the `beta_uncertain` coefficient across three nested models.
- Stays negative & `p < 0.05` after adding disease/size → uncertainty genuinely hurts IoU.
- Shrinks toward 0 & `p > 0.05` once disease is added → the apparent effect was the cardiomegaly/disease mix.

In [ ]:
# --- deps (statsmodels ships with Colab, but ensure it's present) ---
import importlib, subprocess, sys
for pkg in ["statsmodels"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import os, numpy as np, pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
print("ok")

In [ ]:
# --- locate the per-pair results CSV ---
# Priority: (1) Drive, (2) local /content, (3) manual upload.
RESULTS_CSV = None
CANDIDATES = [
    "/content/drive/MyDrive/mimic_ils_rosalia/rosalia_mimic_ils_per_pair.csv",
    "/content/mimic_ils_rosalia/rosalia_mimic_ils_per_pair.csv",
    "rosalia_mimic_ils_per_pair.csv",
]

# try mounting Drive (ignore if not in Colab)
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
except Exception as e:
    print("[info] Drive not mounted:", e)

for c in CANDIDATES:
    if os.path.exists(c):
        RESULTS_CSV = c; break

if RESULTS_CSV is None:
    try:
        from google.colab import files
        print("Upload rosalia_mimic_ils_per_pair.csv ...")
        up = files.upload()
        RESULTS_CSV = list(up.keys())[0]
    except Exception as e:
        raise FileNotFoundError(
            "Could not find rosalia_mimic_ils_per_pair.csv. Set RESULTS_CSV manually."
        ) from e

OUT_DIR = os.path.dirname(RESULTS_CSV) or "."
print("RESULTS_CSV =", RESULTS_CSV)

In [ ]:
# --- load & prepare the regression frame ---
res = pd.read_csv(RESULTS_CSV)
print("rows:", len(res), "| columns:", list(res.columns))

P = res[(res.get("polarity") == "positive") & res["iou"].notna()].copy()
P["is_uncertain"] = (P["uncertainty_label"] == "uncertain").astype(int)
P["log_area"]     = np.log1p(P["gt_px"])          # silver-mask area (skewed -> log)
P["disease"]      = P["target"].astype("category")
# keep only labeled rows (drop 'unknown'/parse failures)
P = P[P["uncertainty_label"].isin(["certain", "uncertain"])].copy()

# tidy CSV of exactly the columns described: disease, area, uncertainty, iou
tidy = P[["target", "gt_px", "log_area", "uncertainty_label", "is_uncertain", "iou"]
         + (["split"] if "split" in P.columns else [])].copy()
tidy_path = os.path.join(OUT_DIR, "iou_regression_table.csv")
tidy.to_csv(tidy_path, index=False)
print("saved tidy table ->", tidy_path, "| n =", len(P))

print("\ncounts by lesion x uncertainty:")
print(pd.crosstab(P["target"], P["uncertainty_label"]).to_string())

In [ ]:
# --- nested OLS models (robust HC3 standard errors) ---
# Watch beta_uncertain change as controls are added.
def fit_report(df, tag):
    m1 = smf.ols("iou ~ is_uncertain", data=df).fit(cov_type="HC3")
    m2 = smf.ols("iou ~ is_uncertain + C(disease)", data=df).fit(cov_type="HC3")
    m3 = smf.ols("iou ~ is_uncertain + C(disease) + log_area", data=df).fit(cov_type="HC3")
    print(f"\n===== {tag}  (n={len(df)}) =====")
    rows = []
    for name, m in [("M1: unc only", m1), ("M2: +disease", m2), ("M3: +disease+size", m3)]:
        b = m.params["is_uncertain"]; p = m.pvalues["is_uncertain"]
        lo, hi = m.conf_int().loc["is_uncertain"]
        print(f"{name:20s} beta_uncertain={b:+.4f}  p={p:.4g}  95%CI=[{lo:+.3f},{hi:+.3f}]  R2={m.rsquared:.3f}")
        rows.append(dict(model=name, beta=b, p=p, lo=lo, hi=hi, r2=m.rsquared))
    return m1, m2, m3, pd.DataFrame(rows)

m1, m2, m3, coefs_all = fit_report(P, "ALL positive findings")

In [ ]:
# --- full model summary (M3) ---
print(m3.summary())

In [ ]:
# --- robustness: fractional logit (IoU is a proportion in [0,1]) ---
# OLS can predict outside [0,1]; a Binomial/logit GLM respects the bounds.
try:
    glm = smf.glm("iou ~ is_uncertain + C(disease) + log_area", data=P,
                  family=sm.families.Binomial()).fit(cov_type="HC3")
    b = glm.params["is_uncertain"]; p = glm.pvalues["is_uncertain"]
    print(f"GLM (fractional logit)  beta_uncertain(log-odds)={b:+.4f}  p={p:.4g}")
    print("(Sign & significance should agree with OLS M3.)")
except Exception as e:
    print("[warn] GLM failed (IoU may contain exact 0/1):", e)

In [ ]:
# --- confound check: exclude cardiomegaly ---
P_nc = P[P["target"] != "cardiomegaly"].copy()
_ = fit_report(P_nc, "EXCLUDING cardiomegaly")

In [ ]:
# --- held-out TEST split only (clean, uncontaminated) ---
if "split" in P.columns and (P["split"] == "test").any():
    _ = fit_report(P[P["split"] == "test"].copy(), "TEST split only")
else:
    print("[info] no 'split' column / no test rows in this results file.")

In [ ]:
# --- coefficient plot: beta_uncertain across the 3 nested models (ALL findings) ---
fig, ax = plt.subplots(figsize=(6, 4))
y = np.arange(len(coefs_all))[::-1]
ax.errorbar(coefs_all["beta"], y,
            xerr=[coefs_all["beta"] - coefs_all["lo"], coefs_all["hi"] - coefs_all["beta"]],
            fmt="o", capsize=4, color="tab:blue")
ax.axvline(0, color="red", ls="--", lw=1)
ax.set_yticks(y); ax.set_yticklabels(coefs_all["model"])
ax.set_xlabel("beta_uncertain (effect on IoU)  —  negative = uncertainty lowers IoU")
ax.set_title("Uncertainty effect on IoU as confounds are added")
for _, r in coefs_all.iterrows():
    ax.annotate(f"p={r['p']:.2g}", (r["beta"], y[list(coefs_all['model']).index(r['model'])]),
                textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
plt.tight_layout()
plot_path = os.path.join(OUT_DIR, "iou_regression_coefplot.png")
plt.savefig(plot_path, dpi=130); plt.show()
print("saved ->", plot_path)

## How to read the result

- **`beta_uncertain`** = change in IoU for *uncertain* vs *certain* findings, holding the other terms constant.
- **M1 → M2 → M3**: if the coefficient collapses toward 0 (and `p` rises above 0.05) when `C(disease)` is added, the naive uncertainty effect was really disease mix (cardiomegaly). If it stays negative and significant, uncertainty independently lowers IoU.
- The **cardiomegaly-excluded** and **test-only** blocks are robustness checks on the same coefficient.
- The **GLM** is a bounds-respecting robustness check; sign/significance should match OLS.

Outputs saved next to the results CSV: `iou_regression_table.csv`, `iou_regression_coefplot.png`.